# Build a Small Language Model from Scratch — on your own stories

Follows the Vizuara notebook step for step. **Step 4, the model architecture,
is copied from it unchanged** — that is the cell the video explains.

Same GPT-2 tokenizer as the video. Five things differ, each marked `# CHANGED:`
in the cell where it happens:

| | video | here | why |
|---|---|---|---|
| dataset | TinyStories | your cleaned JSONL | |
| `block_size` | 128 | 512 | your stories are ~350 tokens; at 128 the model never sees a hook and its payoff together |
| story separator | none | `<\|endoftext\|>` | lets you generate a *fresh* story instead of priming with text |
| dropout | 0.1 | 0.0 | you have far more tokens than parameters |
| checkpoints | model only | full state | Colab disconnects mid-run; this resumes |

Four bugs in the original are also fixed, marked `# BUGFIX:`. The largest:
`min_lr` was set *above* `learning_rate`, so the learning rate rose through
training instead of decaying.

## Step 0: Settings

`DATA_PATH` is your cleaned stories file — one JSON object per line with a
`story_text` field. If your file names that field differently, change it in
Step 1.

On Colab, upload the file to Drive and point at it:
`drive.mount('/content/drive')`, then
`DATA_PATH = "/content/drive/MyDrive/stories.jsonl"`.

In [ ]:
import os

DATA_PATH = "stories.jsonl"
OUT_DIR   = "slm_out"

BLOCK_SIZE = 512      # CHANGED: was 128
DROPOUT    = 0.0      # CHANGED: was 0.1
EPOCHS     = 2.0      # passes over your data
MAX_HOURS  = 11.0     # checkpoint and stop before Colab kills the session
LIMIT      = 0        # stories to use; 0 = all. Set 50000 for a local test run.

# Parameter counts below use the GPT-2 vocabulary. "30m" is the video's config.
SIZES = {
    "30m": dict(n_layer=6, n_head=6,  n_embd=384),   # 30.1M
    "45m": dict(n_layer=6, n_head=8,  n_embd=512),   # 44.9M
    "50m": dict(n_layer=8, n_head=8,  n_embd=512),   # 51.2M
    "75m": dict(n_layer=9, n_head=10, n_embd=640),   # 76.7M
}
MODEL_SIZE = "50m"
N_LAYER = SIZES[MODEL_SIZE]["n_layer"]
N_HEAD  = SIZES[MODEL_SIZE]["n_head"]
N_EMBD  = SIZES[MODEL_SIZE]["n_embd"]

os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
!pip install -q tiktoken datasets

## Step 1: Import the dataset

The video calls `load_dataset("roneneldan/TinyStories")`, which hands back a
DatasetDict of splits. Same thing here, pointed at your local JSONL.

`load_dataset` reads straight into Arrow on disk rather than into Python
memory, so the corpus never sits in RAM as strings. That is what lets the
video's Step 2 run unchanged at your scale.

99% train, 1% validation, shuffled with a fixed seed so the split is identical
every run.

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files=DATA_PATH, split="train")
if LIMIT:
    ds = ds.select(range(LIMIT))

ds = ds.train_test_split(test_size=0.01, seed=42, shuffle=True)
ds["val"] = ds.pop("test")      # two splits: train and val

print(ds)

## Step 2: Tokenize the dataset

GPT-2 tokenizer, as in the video, and this cell is the video's Step 2 almost
line for line: `process()` returning `ids` and `len`, `ds.map(..., num_proc=8)`
to tokenize in parallel, the lengths summed to size the file,
`np.memmap(mode="w+", shape=(arr_len,))` pre-allocated, then filled by sharding
the dataset into 1,024 contiguous batches, and flushed.

`np.memmap` must be told its final length up front. That is the entire reason
the video carries a `len` field, and why tokenizing has to finish before
writing starts.

**CHANGED: `<|endoftext|>` is appended after each story.** The video's
`process()` drops nanoGPT's `ids.append(enc.eot_token)`, so its stories run
together with no boundary, which is why its inference has to be primed with
`"Once upon a time..."`. With the separator the model learns where a story
starts and ends, and you can ask it for a fresh one.

`ds.map` writes the token ids to an Arrow cache on disk, not to RAM. That
matters at this scale: 850M ids as Python ints would be ~6.8 GB of pointers
before the int objects themselves.

In [ ]:
import numpy as np, tiktoken
from tqdm.auto import tqdm

enc = tiktoken.get_encoding("gpt2")
eot_id = enc.eot_token
vocab_size = 50257

def process(example):
    ids = enc.encode_ordinary(example['story_text']) # encode_ordinary ignores any special tokens
    ids.append(eot_id)                               # CHANGED: story boundary
    out = {'ids': ids, 'len': len(ids)}
    return out

if not os.path.exists(f"{OUT_DIR}/train.bin"):
    tokenized = ds.map(
        process,
        remove_columns=ds['train'].column_names,
        desc="tokenizing the splits",
        num_proc=8,
        )
    # concatenate all the ids in each dataset into one large file we can use for training
    for split, dset in tokenized.items():
        arr_len = np.sum(dset['len'], dtype=np.uint64)
        filename = f'{OUT_DIR}/{split}.bin'
        dtype = np.uint16 # (can do since enc.max_token_value == 50256 is < 2**16)
        arr = np.memmap(filename, dtype=dtype, mode='w+', shape=(arr_len,))
        total_batches = min(1024, len(dset))  # a split with fewer rows than shards cannot be sharded

        idx = 0
        for batch_idx in tqdm(range(total_batches), desc=f'writing {filename}'):
            # Batch together samples for faster write
            batch = dset.shard(num_shards=total_batches, index=batch_idx, contiguous=True).with_format('numpy')
            arr_batch = np.concatenate(batch['ids'])
            # Write into mmap
            arr[idx : idx + len(arr_batch)] = arr_batch
            idx += len(arr_batch)
        arr.flush()

# Read back from the files, so this is right whether or not the block above ran.
train_tokens = os.path.getsize(f"{OUT_DIR}/train.bin") // 2
val_tokens   = os.path.getsize(f"{OUT_DIR}/val.bin") // 2
print(f"train.bin: {train_tokens:,} tokens")
print(f"val.bin:   {val_tokens:,} tokens")

## Step 3: Create input–output batches

Unchanged from the video (which takes it from nanoGPT). Random `block_size`
windows out of the packed stream; the memmap is reopened every call because
holding one open across thousands of iterations leaks memory.

In [ ]:
import torch

if torch.cuda.is_available():
    device = "cuda"
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
device_type = device

def get_batch(split):
    data = np.memmap(f"{OUT_DIR}/{split}.bin", dtype=np.uint16, mode="r")
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])
    if device_type == "cuda":
        x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)
    return x, y

print("device:", device)

## Step 4: Define the SLM model architecture

**This cell is copied from the Vizuara notebook unchanged.** LayerNorm, causal
self-attention, a 4x GELU MLP, learned position embeddings, and weight tying
between the input embedding and the output head.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass
import numpy as np
from tqdm.auto import tqdm
from contextlib import nullcontext
import os

class LayerNorm(nn.Module):
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None
    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.flash = hasattr(F, 'scaled_dot_product_attention')
        if not self.flash:
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                       .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        if self.flash:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.attn_dropout.p if self.training else 0.0, is_causal=True)
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)
    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = LayerNorm(config.n_embd, config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln2 = LayerNorm(config.n_embd, config.bias)
        self.mlp = MLP(config)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

@dataclass
class GPTConfig:
    block_size: int
    vocab_size: int
    n_layer: int
    n_head: int
    n_embd: int
    dropout: float = 0.0
    bias: bool = True

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd),
            wpe=nn.Embedding(config.block_size, config.n_embd),
            drop=nn.Dropout(config.dropout),
            h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f=LayerNorm(config.n_embd, config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight  # weight tying

        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size
        pos = torch.arange(0, t, dtype=torch.long, device=device)

        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
            return logits, loss
        else:
            logits = self.lm_head(x[:, [-1], :])
            return logits, None

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        Generate tokens given a conditioning sequence.
        idx: Tensor of shape (B, T)
        """
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx



In [ ]:
config = GPTConfig(
    vocab_size=vocab_size,
    block_size=BLOCK_SIZE,   # CHANGED: 512, not 128
    n_layer=N_LAYER,
    n_head=N_HEAD,
    n_embd=N_EMBD,
    dropout=DROPOUT,         # CHANGED: 0.0, not 0.1
    bias=True,
)
model = GPT(config).to(device)

n_params = sum(p.numel() for p in model.parameters())
n_emb = model.transformer.wte.weight.numel() + model.transformer.wpe.weight.numel()
print(f"{n_params/1e6:.1f}M parameters "
      f"({n_emb/1e6:.1f}M embeddings, {(n_params-n_emb)/1e6:.1f}M transformer)")

## Step 5: Define the loss function

**BUGFIX.** The video uses `eval_iters` as *both* the evaluation interval and
the number of evaluation batches, so with `eval_iters=500` and
`max_iters=20000` it runs 40 evaluations x 500 batches x 2 splits = 40,000
evaluation passes against 20,000 training steps — more compute spent measuring
than learning. They are separate settings here.

In [ ]:
@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ("train", "val"):
        losses = torch.zeros(EVAL_BATCHES)
        for k in range(EVAL_BATCHES):
            X, Y = get_batch(split)
            with ctx:
                _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

## Step 6: Training configuration

**BUGFIX: `min_lr` must be BELOW `learning_rate`.** The video sets
`learning_rate=1e-4` and `min_lr=5e-4`. `CosineAnnealingLR(eta_min=5e-4)` then
inverts — the schedule climbs 1e-4 -> 5e-4 monotonically across all 20,000
iterations instead of decaying.

`max_iters` comes from `EPOCHS` rather than being hard-coded. The video's
20,000 x 32 x 128 = 82M tokens is 0.17 of one pass over TinyStories.

In [ ]:
from contextlib import nullcontext

batch_size = 32
block_size = BLOCK_SIZE
gradient_accumulation_steps = 4      # 32 x 4 x 512 = 65,536 tokens per update

learning_rate = 1e-3
min_lr        = 1e-4                 # BUGFIX: below the peak, so cosine DECAYS
grad_clip     = 1.0
EVAL_BATCHES  = 50                   # BUGFIX: separate from the eval interval

tokens_per_update = batch_size * gradient_accumulation_steps * block_size
total_updates = max(1, int(EPOCHS * train_tokens / tokens_per_update))
max_iters     = total_updates * gradient_accumulation_steps

# Scaled to the run length: fixed values break a short run, where a warmup
# longer than the run means the rate never decays and an eval interval longer
# than the run means no checkpoint is ever written.
warmup_steps  = min(500, max(1, total_updates // 20))
EVAL_INTERVAL = min(500, max(1, total_updates // 10))

if device_type == "cuda":
    # T4 and P100 are pre-Ampere and have NO bf16, so check rather than assume.
    dtype = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"
elif device_type == "mps":
    dtype = "bfloat16"
else:
    dtype = "float32"
ptdtype = {"float32": torch.float32, "bfloat16": torch.bfloat16, "float16": torch.float16}[dtype]
ctx = (nullcontext() if dtype == "float32"
       else torch.amp.autocast(device_type=device_type, dtype=ptdtype))

torch.manual_seed(42)

print(f"dtype={dtype}")
print(f"{tokens_per_update:,} tokens/update, {total_updates:,} updates")
print(f"warmup {warmup_steps:,}, evaluating every {EVAL_INTERVAL:,}")

## Step 7: Optimizer and scheduler

**BUGFIX: the scheduler steps once per *optimizer* update, not once per
micro-batch.** The video calls `scheduler.step()` every iteration while the
optimizer only steps every 32, so its "1000-step warmup" is really about 31
weight updates.

In [ ]:
from torch.optim.lr_scheduler import LinearLR, SequentialLR, CosineAnnealingLR

def build_optimizer():
    opt = torch.optim.AdamW(model.parameters(), lr=learning_rate,
                            betas=(0.9, 0.95), weight_decay=0.1, eps=1e-9)
    sch = SequentialLR(opt, schedulers=[
        LinearLR(opt, total_iters=warmup_steps),
        CosineAnnealingLR(opt, T_max=max(total_updates - warmup_steps, 1), eta_min=min_lr)],
        milestones=[warmup_steps])
    sc = torch.amp.GradScaler("cuda" if device_type == "cuda" else "cpu",
                              enabled=(dtype == "float16" and device_type == "cuda"))
    return opt, sch, sc

# Confirm the schedule decays before spending hours on it.
optimizer, scheduler, scaler = build_optimizer()
probe = []
for _ in range(total_updates):
    probe.append(optimizer.param_groups[0]["lr"]); optimizer.step(); scheduler.step()
print(f"lr: start {probe[0]:.2e} -> peak {max(probe):.2e} -> end {probe[-1]:.2e}")
assert probe[-1] < max(probe), "learning rate does not decay -- check min_lr"

optimizer, scheduler, scaler = build_optimizer()   # rebuild clean

## Step 8: Pre-train the SLM

**CHANGED: full training state is checkpointed, and re-running this cell
resumes.** The video saves `model.state_dict()` only, which cannot resume —
without the optimizer's momentum the loss jumps when you restart. Colab *will*
disconnect during a run this long.

In [ ]:
import time

CKPT = f"{OUT_DIR}/ckpt.pt"
BEST = f"{OUT_DIR}/best_model_params.pt"

start_iter, best_val_loss, history = 0, float("inf"), []

if os.path.exists(CKPT):
    ck = torch.load(CKPT, map_location=device, weights_only=False)
    model.load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    scheduler.load_state_dict(ck["scheduler"])
    if ck.get("scaler"): scaler.load_state_dict(ck["scaler"])
    start_iter, best_val_loss, history = ck["iter"], ck["best_val_loss"], ck["history"]
    print(f"resumed at iter {start_iter:,}/{max_iters:,}, best val {best_val_loss:.4f}")

def save(path):
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict() if scaler.is_enabled() else None,
                "iter": it + 1, "best_val_loss": best_val_loss,
                "history": history, "config": config.__dict__}, path + ".tmp")
    os.replace(path + ".tmp", path)   # atomic: a kill mid-write keeps the old one

t0 = time.time()
model.train()
optimizer.zero_grad(set_to_none=True)

for it in tqdm(range(start_iter, max_iters), initial=start_iter, total=max_iters):
    X, y = get_batch("train")
    with ctx:
        logits, loss = model(X, y)
        loss = loss / gradient_accumulation_steps
    scaler.scale(loss).backward()      # outside ctx: backward is not autocast

    if (it + 1) % gradient_accumulation_steps == 0:
        scaler.unscale_(optimizer)     # clip real gradients, not scaled ones
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer); scaler.update()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()               # BUGFIX: per optimizer step

        update = (it + 1) // gradient_accumulation_steps
        if update % EVAL_INTERVAL == 0:
            L = estimate_loss(model)
            print(f"update {update:,}: train {L['train']:.4f}  val {L['val']:.4f}  "
                  f"lr {optimizer.param_groups[0]['lr']:.2e}")
            history.append({"update": update, **L})
            if L["val"] < best_val_loss:
                best_val_loss = L["val"]
                torch.save(model.state_dict(), BEST)
                print(f"  new best val {best_val_loss:.4f}")
            save(CKPT)

        if MAX_HOURS and (time.time() - t0) > MAX_HOURS * 3600:
            save(CKPT)
            print(f"MAX_HOURS reached at update {update:,} -- re-run this cell to continue.")
            break

save(CKPT)
print(f"done: iter {it+1:,}/{max_iters:,}, {(time.time()-t0)/3600:.2f}h, "
      f"best val {best_val_loss:.4f}")

## Step 9: Plot the loss

In [ ]:
import matplotlib.pyplot as plt

x = [h["update"] for h in history]
plt.plot(x, [h["train"] for h in history], label="train")
plt.plot(x, [h["val"] for h in history], label="val")
plt.xlabel("optimizer update"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

## Step 10: Generate stories

The video primes with `"Once upon a time there was a pumpkin."` because its
model has no story-boundary token. Yours does, so priming with
`<|endoftext|>` asks for a **completely fresh story**, and generation stops
when the model decides the story is over.

In [ ]:
model.load_state_dict(torch.load(BEST, map_location=device))
model.eval()

@torch.no_grad()
def write_stories(n=5, max_new_tokens=600, temperature=0.8, top_k=200, prompt=None):
    ids = [eot_id] + (enc.encode_ordinary(prompt) if prompt else [])
    idx = torch.tensor([ids] * n, dtype=torch.long, device=device)
    done = torch.zeros(n, dtype=torch.bool, device=device)
    for _ in range(max_new_tokens):
        logits, _ = model(idx[:, -config.block_size:])
        logits = logits[:, -1, :] / temperature
        v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
        logits[logits < v[:, [-1]]] = -float("Inf")
        nxt = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
        nxt = torch.where(done.unsqueeze(1), torch.full_like(nxt, eot_id), nxt)
        done |= nxt.squeeze(1) == eot_id
        idx = torch.cat((idx, nxt), dim=1)
        if bool(done.all()): break

    for row in idx.tolist():
        out = row[1:]
        ended = eot_id in out
        if ended: out = out[:out.index(eot_id)]
        text = enc.decode(out).strip()
        print(f"=== {len(text.split())} words, {'complete' if ended else 'TRUNCATED'} ===")
        print(text)
        print()

write_stories(5)

In [ ]:
write_stories(2, prompt='"You think you can just walk in here and take what is ours?"')

## Save before the session dies

Colab wipes the container. Download `best_model_params.pt` — that plus the
GPT-2 tokenizer (which `tiktoken` fetches anywhere) is all you need to generate
stories on your own machine.

Keep `ckpt.pt` too if you want to carry on training in a later session.

In [ ]:
for f in sorted(os.listdir(OUT_DIR)):
    print(f"  {f:28s} {os.path.getsize(OUT_DIR + '/' + f)/1e6:8.2f} MB")

# Colab: from google.colab import files; files.download(BEST)